In [ ]:
import torch
import torch.nn as nn
import pandas as pd
from sklearn.model_selection import train_test_split
from collections import Counter
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from sklearn.metrics import accuracy_score
import os
from transformers import get_cosine_schedule_with_warmup
import torch.autograd as autograd
from sklearn.utils.class_weight import compute_class_weight

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False


os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
df = pd.read_csv(r'/kaggle/input/datasets/popchic/normik/train.csv')

train_df, temp_df = train_test_split(
    df, test_size=0.25, stratify=df["language"], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.4, stratify=temp_df["language"], random_state=42
)

In [ ]:
class NLIDataset(Dataset):
    def __init__(self, premise, hypothesis, tokenizer, label = None):
        self.premise = list(premise)
        self.hypothesis = list(hypothesis)
        self.tokenizer = tokenizer
        self.label = list(label) if label is not None else label 
        
    def __len__(self): 
        return len(self.premise)
    def __getitem__(self, i):
        enc = self.tokenizer(self.premise[i], self.hypothesis[i], truncation=True,
                        padding="max_length", max_length=128, return_tensors="pt")
        item = {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0)
        }
        if self.label is not None:
            item["label"] = torch.tensor(self.label[i], dtype=torch.long)
        return item

BATCH_SIZE = 32

def get_loaders(tokenizer, max_length=128, bs=32):
    tr_ds = NLIDataset(train_df["premise"], train_df["hypothesis"], tokenizer, train_df["label"])
    vl_ds = NLIDataset(val_df["premise"], val_df["hypothesis"], tokenizer, val_df["label"])
    return (
        DataLoader(tr_ds, batch_size=bs, shuffle=True, pin_memory=True),
        DataLoader(vl_ds, batch_size=bs, shuffle=False, pin_memory=True)
    )

In [ ]:
def train(model_name, name, num_labels=3, batch_size=32, max_length=128, epochs=4):
    autograd.set_detect_anomaly(True)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name,
                                                               num_labels=3,trust_remote_code=True)
    model = model.to(device)
    model = model.to(torch.float32)

    train_loader, val_loader = get_loaders(tokenizer, 128, 32)
    
    classes = np.unique(train_df["label"])
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=train_df["label"])
    criterion = torch.nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32), label_smoothing=0.1)
    criterion = criterion.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=0.01)
    total_steps = len(train_loader) * epochs
    scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(0.05 * total_steps), num_training_steps=total_steps)
    
    best_val_acc = 0.0
    
    for epoch in range(epochs):
        model.train()
        total_train_loss = 0.0
        train_preds, train_labels_list = [], []
        for batch in tqdm(train_loader, desc=f"Epoch {epoch + 1} | Train"):
            optimizer.zero_grad()
            ids  = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            logits = model(input_ids=ids, attention_mask=mask).logits
            train_preds.append(torch.argmax(logits, dim=-1).cpu())
            train_labels_list.append(labels.cpu())
    
            loss = criterion(logits, labels)
            total_train_loss += loss.item()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
    
        train_acc = accuracy_score(torch.cat(train_labels_list), torch.cat(train_preds))
        
        model.eval()
        total_val_loss = 0
        val_preds, val_labels = [], []
        
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoch_val {epoch+1}"):
                ids  = batch["input_ids"].to(device)
                mask = batch["attention_mask"].to(device)
                labels = batch["label"].to(device)
                logits = model(input_ids=ids, attention_mask=mask).logits
                
                loss = criterion(logits, labels)
                total_val_loss += loss.item()
                
                preds = torch.argmax(logits, dim=-1)
                val_preds.append(preds.cpu())
                val_labels.append(labels.cpu())
        
        val_acc = accuracy_score(torch.cat(val_labels), torch.cat(val_preds))
        print(f"Epoch {epoch+1} | "
              f"Train Loss: {total_train_loss/len(train_loader):.4f} | Train Acc: {train_acc:.4f} | "
              f"Val Loss: {total_val_loss/len(val_loader):.4f} | Val Acc: {val_acc:.4f}")
    
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f"{name}_best_model.pth")
            print("Новая лучшая модель сохранена в память!")
    del model, optimizer, scheduler, train_loader, val_loader, tokenizer
    torch.cuda.empty_cache()

In [ ]:
train("joeddav/xlm-roberta-large-xnli", "fst")
train("MoritzLaurer/mdeberta-v3-base-xnli-multilingual-nli-2mil7", "snd")

tok_snd = AutoTokenizer.from_pretrained("MoritzLaurer/mdeberta-v3-base-xnli-multilingual-nli-2mil7")
tok_fst = AutoTokenizer.from_pretrained("joeddav/xlm-roberta-large-xnli")

model_snd = AutoModelForSequenceClassification.from_pretrained("MoritzLaurer/mdeberta-v3-base-xnli-multilingual-nli-2mil7", num_labels=3, trust_remote_code=True).to(device).eval()
model_snd.load_state_dict(torch.load("snd_best_model.pth", weights_only=True))

model_fst = AutoModelForSequenceClassification.from_pretrained("joeddav/xlm-roberta-large-xnli", num_labels=3, trust_remote_code=True).to(device).eval()
model_fst.load_state_dict(torch.load("fst_best_model.pth", weights_only=True))

_, val_loader_snd = get_loaders(tok_snd)
_, val_loader_fst = get_loaders(tok_fst)

def get_val_logits(model, loader):
    logits = []
    with torch.no_grad():
        for b in tqdm(loader, desc="Inference Val"):
            logits.append(model(input_ids=b["input_ids"].to(device), attention_mask=b["attention_mask"].to(device)).logits.cpu())
    return torch.cat(logits)

log_snd = get_val_logits(model_snd, val_loader_snd)
log_fst = get_val_logits(model_fst, val_loader_fst)
val_y_true = np.array([b["label"].cpu().numpy() for b in val_loader_snd.dataset])

best_w, best_acc = 0.5, 0.0
for w in np.arange(0.0, 1.01, 0.05):
    ens_logits = w * log_snd + (1 - w) * log_fst
    acc = accuracy_score(val_y_true, ens_logits.argmax(dim=-1).numpy())
    if acc > best_acc:
        best_acc, best_w = acc, w

print(f"Оптимальный вес DeBERTa: {best_w:.2f} | XLM-R: {1-best_w:.2f} | Val Acc: {best_acc:.4f}")

def get_test_logits(model, loader):
    preds = []
    with torch.no_grad():
        for b in tqdm(loader, desc="Inference Test"):
            preds.append(model(input_ids=b["input_ids"].to(device), attention_mask=b["attention_mask"].to(device)).logits.cpu())
    return torch.cat(preds)

test_loader_snd = DataLoader(NLIDataset(test_df["premise"], test_df["hypothesis"], tok_snd, test_df["label"]), batch_size=16, shuffle=False)
test_loader_fst = DataLoader(NLIDataset(test_df["premise"], test_df["hypothesis"], tok_fst, test_df["label"]), batch_size=16, shuffle=False)

logits_snd, labels_true = [], []
with torch.no_grad():
    for b in tqdm(test_loader_snd, desc="Test DeBERTa"):
        logits = model_snd(input_ids=b["input_ids"].to(device), attention_mask=b["attention_mask"].to(device)).logits.cpu()
        logits_snd.append(logits)
        labels_true.append(b["label"].cpu())

logits_fst = []
with torch.no_grad():
    for b in tqdm(test_loader_fst, desc="Test XLM-R"):
        logits_fst.append(model_fst(input_ids=b["input_ids"].to(device), attention_mask=b["attention_mask"].to(device)).logits.cpu())


logits_snd = torch.cat(logits_snd)
logits_fst = torch.cat(logits_fst)
y_true = torch.cat(labels_true).numpy()

ensemble_logits = best_w * logits_snd + (1 - best_w) * logits_fst
y_pred = ensemble_logits.argmax(dim=-1).numpy()
test_acc = accuracy_score(y_true, y_pred)
print(f"✅ Test Accuracy: {test_acc:.4f}")

In [ ]:
test_df = pd.read_csv(r'/kaggle/input/datasets/popchic/normik/test.csv')
test_loader_snd = DataLoader(NLIDataset(test_df["premise"], test_df["hypothesis"], tok_snd), batch_size=16, shuffle=False)
test_loader_fst = DataLoader(NLIDataset(test_df["premise"], test_df["hypothesis"], tok_fst), batch_size=16, shuffle=False)
    
test_log_snd = get_test_logits(model_snd, test_loader_snd)
test_log_fst = get_test_logits(model_fst, test_loader_fst)

ensemble_logits = best_w * test_log_snd + (1 - best_w) * test_log_fst
final_preds = ensemble_logits.argmax(dim=-1).numpy()

sub = pd.DataFrame({"id": test_df["id"], "prediction": final_preds})
sub.to_csv("submission_ensemble.csv", index=False)
print(f"Submission saved: {len(sub)} rows. Head:\n{sub.head()}")